# Pipeline, stages 3 to 8

Picks up from `pipeline_stages_1_2.ipynb`, which produced `topics.json`
(big move days with their retrieved documents).

- Stage 3: pull candidate cause events out of the documents
- Stage 4: score them with three models, route disagreements to a human
- Stage 5: assemble four-option questions
- Stage 6: quality gates
- Stage 7: human review and agreement
- Stage 8: splits and final EDA

Needs API keys for the three model providers.

In [ ]:
!pip install -q openai anthropic google-generativeai scikit-learn

import os
# Set whichever you have. Three DIFFERENT families matters: models from the
# same family share biases and would agree for the wrong reasons.
os.environ["OPENAI_API_KEY"]    = ""
os.environ["ANTHROPIC_API_KEY"] = ""
os.environ["GOOGLE_API_KEY"]    = ""

from llm import available_models
print("models available:", available_models())

In [ ]:
import json, random
import extraction, scoring, assembly, quality, finalise

with open("topics.json") as f:
    topics = json.load(f)
print(f"{len(topics)} topics loaded")

## Stage 3: extract candidate causes

Following CRAB, a model reads each article and lists the events it reports,
rather than using a syntactic extractor. CRAB found this gives better
precision and events at the right level of abstraction.

Each event keeps its source document and date, which is what makes the
temporal distractors possible later.

In [ ]:
# Start with one topic to check the output looks sane before spending
# money on all of them.
t = extraction.extract_candidates_for_topic(topics[0], model="claude")

print(f"\n{len(t['candidates'])} candidates for {t['asset']} on {t['event_date']}\n")
for cand in t["candidates"][:12]:
    print(f"  [{cand['position']:7s}] {cand['text'][:95]}")

In [ ]:
from checkpoint import resumable_map, checkpoint_status

RUN_ALL_EXTRACTION = False

if RUN_ALL_EXTRACTION:
    extracted = resumable_map(
        items   = topics,
        key_fn  = lambda t: f"{t['asset']}_{t['event_date']}",
        work_fn = lambda t: extraction.extract_candidates_for_topic(t, model="gemini"),
        path    = "extracted_progress.json",
    )
    with open("extracted.json","w") as f:
        json.dump(extracted, f, indent=2)
else:
    extracted = [t]

## Stage 4: score with three models

Every candidate is rated 0 to 3 by each model. Where they agree the label
is settled; where they disagree it goes to a human.

Candidates dated after the target event are given 0 without an API call,
since something that happened afterwards cannot have caused it. They stay
in the pool because they make good temporal distractors.

Expect roughly a quarter of candidates to need review. If it is far higher,
the scoring prompt is probably ambiguous and worth tightening before
spending annotator time.

In [ ]:
# The target event text describes the price move itself.
for t in extracted:
    if not t.get("target_event"):
        t["target_event"] = f"{t['asset']} moved sharply on {t['event_date']}."

# This is the stage that costs money, so checkpointing matters most here.
# A dropped session should never mean paying for the same API calls twice.
scored = resumable_map(
    items   = extracted,
    key_fn  = lambda t: f"{t['asset']}_{t['event_date']}",
    work_fn = lambda t: scoring.score_topic(t, models=["gemini"]),
    path    = "scored_progress.json",
)

with open("scored.json","w") as f:
    json.dump(scored, f, indent=2, default=str)

In [ ]:
queue = scoring.review_queue(scored)
print(f"{len(queue)} candidates need human review\n")
for item in queue[:8]:
    print(f"  spread {item['spread']}  {item['model_scores']}")
    print(f"    {item['candidate'][:95]}")

## Stage 5: assemble the questions

Correct options are anything scored 2 or 3. Distractors are stratified into
the three AER types (temporal, semantic, background) and chosen so their
lengths sit close to the correct options.

A question is dropped rather than shipped if the length gap cannot be
closed. Dropping is much cheaper than shipping a leaky item.

In [ ]:
rng = random.Random(42)

questions = []
for t in scored:
    q = assembly.build_question(t, rng=rng)
    if q:
        q["id"] = f"{t['asset']}_{t['event_date']}"
        questions.append(q)

questions = assembly.rebalance_positions(questions, rng)

print(f"{len(questions)} questions built from {len(scored)} topics "
      f"({len(scored)-len(questions)} dropped)")

check = assembly.verify_consistency(questions)
print(f"label consistency: {check['n_bad']} problems out of {check['n_checked']}")

In [ ]:
# eyeball one
if questions:
    q = questions[0]
    print(q["target_event"], "\n")
    for L in "ABCD":
        mark = "*" if L in q["golden_answer"] else " "
        print(f" {mark} {L}. [{q['causal_strength'][L]}] "
              f"({q['option_types'][L]}) {q[f'option_{L}']}")
    print(f"\n answer: {q['golden_answer']}")

## Stage 6: quality gate

The check that matters most is style leakage. In the EDA on AER, a
classifier reading only the option text scored 89.4% against a 60.6%
baseline, meaning the phrasing gave the answer away. ART was cleaned until
that fell to about 51%.

The gate also looks for the structural shortcuts the winning AER team
exploited for 5.6 free points: the none option always being correct, and
duplicate options sharing a truth value.

**If this fails, go back and rewrite the distractors.** Shipping a batch
that fails the gate defeats the point of the whole design.

In [ ]:
report = quality.run_gate(questions)

In [ ]:
if not report["passed"]:
    print("Batch did not pass. Worst offenders by length gap:\n")
    worst = sorted(questions, key=lambda q: -q.get("length_gap", 0))[:5]
    for q in worst:
        print(f"  gap {q['length_gap']}  {q['id']}")

## Stage 7: human review

Everyone labels a shared subset, which is what Krippendorff's alpha is
computed on, and the rest is divided up. With three annotators and 15%
overlap each person sees about 43% of the queue rather than all of it.

Alpha needs at least two annotators to exist at all, since it measures
agreement between people. For reference: AER reported 0.51, CRAB's expert
reviewers 0.70, UNcommonsense 0.40 to 0.60.

In [ ]:
info = finalise.make_review_sheet(queue, "review_sheet.json",
                                  overlap_frac=0.15, n_annotators=3)
print(f"queue {info['queue_size']}, shared subset {info['overlap_items']}")
print("per annotator:", info["per_annotator"])

In [ ]:
# After everyone fills in their "your_score" fields and saves their copy:
# agreement = finalise.collect_reviews([
#     "review_annotator_1.json", "review_annotator_2.json", "review_annotator_3.json"])
# print(agreement)

## Stage 8: splits and EDA

Split by event, not by question, so no market event appears in more than
one split. Two questions about the same event share documents, so splitting
by question would leak evidence between train and test.

The cardinality mix is kept similar across splits. AER's test set was much
easier than its dev set (18.3% multi-answer against 47.5%), which makes the
two hard to compare.

In [ ]:
splits = finalise.make_splits(questions)

for name, qs in splits.items():
    with open(f"{name}.jsonl", "w") as f:
        for q in qs:
            f.write(json.dumps(q) + "\n")
print("\nwritten train.jsonl, dev.jsonl, test.jsonl")

In [ ]:
finalise.print_eda(finalise.dataset_eda(questions, topics))

## Evaluating models on it

This is what the dataset is for. The headline number matters less than the
breakdown: the winning AER team found under-selections outnumbered
over-selections 1,389 to 52 across 14 models, and that accuracy collapsed
as the number of correct answers rose.

Whether the same pattern shows up in financial reasoning is the finding
worth reporting.

In [ ]:
import evaluate

docs_by_topic = {t["topic_id"]: t for t in topics if "topic_id" in t}

evals = []
for m in available_models():
    print(f"\nevaluating {m}")
    ev = evaluate.evaluate_model(splits["test"], docs_by_topic, model=m)
    evals.append(ev)

if evals:
    print()
    print(evaluate.compare_models(evals))

In [ ]:
for ev in evals:
    fa = evaluate.failure_analysis(ev, splits["test"])
    print(f"\n=== {ev['model']} ===")
    uo = fa["under_vs_over"]
    print(f"  under-selected {uo['under_selected']}, over-selected {uo['over_selected']}")
    print(f"  ({uo['aer_reference']})")
    print("  accuracy by number of correct answers:")
    for k, v in fa["by_cardinality"].items():
        print(f"    {k} correct: exact {v['exact_match']:.2f}, F1 {v['f1']:.2f}, "
              f"predicted {v['mean_predicted']} on average")
    print("  distractors that fooled it:", fa["distractors_that_fooled_it"])